In [1]:
%%capture
%pip install --upgrade kaggle

In [2]:
import subprocess
import json
from IPython.display import HTML, display
import kagglehub
import pandas as pd
from pathlib import Path


/home/grogy/python_playground/simple_ml_project/my_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
result = subprocess.run(
    [
        "kaggle", "competitions", "pages", "list",
        "-c", "titanic",
        "--page-name", "data-description",
        "--content",
        "--format", "json",
        "-q"
    ],
    capture_output=True,
    text=True,
    check=True
)

pages = json.loads(result.stdout)
display(HTML(pages[0]["content"]))

Variable,Definition,Key
survival,Survival,"0 = No, 1 = Yes"
pclass,Ticket class,"1 = 1st, 2 = 2nd, 3 = 3rd"
sex,Sex,
Age,Age in years,
sibsp,# of siblings / spouses aboard the Titanic,
parch,# of parents / children aboard the Titanic,
ticket,Ticket number,
fare,Passenger fare,
cabin,Cabin number,
embarked,Port of Embarkation,"C = Cherbourg, Q = Queenstown, S = Southampton"


In [4]:
path = Path(kagglehub.competition_download("titanic"))

train = pd.read_csv(path / "train.csv")
test = pd.read_csv(path / "test.csv")

In [5]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
display("====== TRAIN INFO ======")
train.info()
display(train.isna().sum().sort_values(ascending=False))
display(train["Survived"].value_counts())
display(train["Survived"].value_counts(normalize=True))
display("====== TEST INFO ======")
test.info()
display(test.isna().sum())

'====== TRAIN INFO ======'

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


None

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

Survived
0    549
1    342
Name: count, dtype: int64

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

'====== TEST INFO ======'

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


None

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [30]:
missing_values = pd.DataFrame({
    "Counts": train.isna().sum(),
    "Prct": (train.isna().mean()*100).round(1)
}).sort_values("Prct", ascending=False)

display(missing_values)

,Counts,Prct
Cabin,687,77.1
Age,177,19.9
Embarked,2,0.2
PassengerId,0,0.0
Name,0,0.0
Pclass,0,0.0
Survived,0,0.0
Sex,0,0.0
Parch,0,0.0
SibSp,0,0.0


In [ ]:
display(train["Age"].value_counts())
display(train["Cabin"].value_counts())
display(train["Embarked"].value_counts())

Age
24.00    30
22.00    27
18.00    26
28.00    25
19.00    25
         ..
24.50     1
0.67      1
0.42      1
34.50     1
74.00     1
Name: count, Length: 88, dtype: int64

Based on data quality in features, i decided, that I will remove feature Cabin, becouse to much values is missing, i will fill Age with median an Embark with mode.

In [ ]:
train_clean = train.copy()
test_clean = test.copy()

train_clean = train_clean.drop(columns=["Cabin"])
test_clean = test_clean.drop(columns=["Cabin"])

train_clean["Age"] = train_clean["Age"].fillna(train_clean["Age"].median())
test_clean["Age"] = test_clean["Age"].fillna(test_clean["Age"].median())

embarked_mode_train = train_clean["Embarked"].mode().iloc[0]
train_clean["Embarked"] = train_clean["Embarked"].fillna(embarked_mode_train)

test_clean["Fare"] = test_clean["Fare"].fillna(test_clean["Fare"].median())

In [28]:
display(train_clean.isna().sum().sort_values(ascending=False))
display(test_clean.isna().sum().sort_values(ascending=False))

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [36]:
train.assign(
    CabinKnown=train["Cabin"].notna()
).groupby("CabinKnown")["Survived"].agg(["count", "mean"])

,count,mean
CabinKnown,,
False,687,0.299854
True,204,0.666667
